# NutriLens - Indian Thali Model Training & Prototype Pipeline
This Kaggle notebook trains the **LRASPP MobileNetV3-Large semantic segmenter** and builds the **OpenCLIP ViT-B/32 prototype recognition index** for NutriLens on the IIIT-H Indian Thali Dataset (ITD).

### Output Artifacts generated:
1. `itd_segmenter.pt` (Trained segmentation checkpoint)
2. `itd_recognition_index.json` & `itd_recognition_index_embeddings.npy` (OpenCLIP dish prototype index)
3. `itd_prototypes_manifest.json`

---

### Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio open-clip-torch opencv-python Pillow numpy

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Step 2: Set Dataset Paths & Class Map
Ensure your dataset is linked to the Kaggle notebook. If uploaded as a Kaggle dataset, adjust `DATASET_ROOT`.

In [ ]:
import os
from pathlib import Path
import json

# Check Kaggle input folder or set custom path
KAGGLE_INPUT = Path("/kaggle/input")
print("Available input directories:", list(KAGGLE_INPUT.glob("*")))

# Set DATASET_DIR pointing to train directory containing 'images' and 'masks'
# e.g. /kaggle/input/indian-thali-dataset/ITD/train or similar
DATASET_ROOT = None
for p in KAGGLE_INPUT.rglob("train"):
    if (p / "images").is_dir() and (p / "masks").is_dir():
        DATASET_ROOT = p
        break

if not DATASET_ROOT:
    # Fallback default
    DATASET_ROOT = Path("/kaggle/working/ITD/train")
    print("Please verify DATASET_ROOT:", DATASET_ROOT)
else:
    print("Found ITD train folder at:", DATASET_ROOT)

OUTPUT_DIR = Path("/kaggle/working/nutrilens_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 51-class IIIT-H ITD Class Map
CLASS_MAP = {
  "0": "background",
  "1": "Aloo Dry fry",
  "2": "Avakaya Muddha Papu Rice",
  "3": "Baby-Corn & Capsicum-Dry",
  "4": "Cabbage Pakodi",
  "5": "Cabbage fry",
  "6": "Capsicum Paneer Curry",
  "7": "Chakar-Pongal",
  "8": "Chole-Masala",
  "9": "Cluster Beans Curry",
  "10": "Cucumber-Raitha",
  "11": "Gobi Masala Curry",
  "12": "Gutti Vankaya Curry",
  "13": "Jeera Rice",
  "14": "Mixed Curry",
  "15": "Muskmelon",
  "16": "Rajma",
  "17": "Rasgulla",
  "18": "Sambar",
  "19": "Tomato Rasam",
  "20": "Vankaya-Ali-Karam",
  "21": "Veg-Biriyani",
  "22": "aloo-curry",
  "23": "curd",
  "24": "dal",
  "25": "fresh-chutney",
  "26": "green-salad",
  "27": "Moong-Beans-Curry",
  "28": "khichdi",
  "29": "lemon-rice",
  "30": "live-roti-with-ghee",
  "31": "non-spicy-curry-bottle-gourd",
  "32": "papad",
  "33": "plain-rice",
  "34": "watermelon",
  "35": "Aloo-Fry",
  "36": "Banana",
  "37": "Mix-Fruit",
  "38": "Non-Spicy-Baby-Corn & Capsicum-Dry",
  "39": "Sweet",
  "40": "Tomato-Rice",
  "41": "fried-papad-rings",
  "42": "gravy",
  "43": "ivy-gourd-fry",
  "44": "mango-pickle",
  "45": "papad-chat",
  "46": "pepper-rasam",
  "47": "pineapple",
  "48": "corn-fry",
  "49": "paneer-curry",
  "50": "gulab-jamun"
}

CLASS_MAP_PATH = OUTPUT_DIR / "iiith_itd_class_map.json"
CLASS_MAP_PATH.write_text(json.dumps(CLASS_MAP, indent=2))
print("Class map saved to:", CLASS_MAP_PATH)

### Step 3: Train LRASPP MobileNetV3-Large Segmenter
Trains the lightweight segmentation model to segment different dishes on the thali.

In [ ]:
import math
import random
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.models.segmentation as seg_models
import torch.nn as nn

IMAGE_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 8
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ITDDataset(Dataset):
    def __init__(self, samples, image_size=IMAGE_SIZE):
        self.samples = samples
        self.image_size = image_size
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        with Image.open(img_path) as im:
            image = im.convert("RGB").resize((self.image_size, self.image_size), Image.BILINEAR)
        with Image.open(mask_path) as mk:
            mask = mk.convert("L").resize((self.image_size, self.image_size), Image.NEAREST)

        img_arr = (np.array(image, dtype=np.float32) / 255.0 - self.mean) / self.std
        img_tensor = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(np.array(mask, dtype=np.int64)).long()
        return img_tensor, mask_tensor

# Gather paired samples
images_dir = DATASET_ROOT / "images"
masks_dir = DATASET_ROOT / "masks"
all_samples = []
for img_file in images_dir.glob("*.jpg"):
    mask_file = masks_dir / f"{img_file.stem}.png"
    if mask_file.exists():
        all_samples.append((img_file, mask_file))

print(f"Total valid paired samples: {len(all_samples)}")
random.seed(42)
random.shuffle(all_samples)
val_count = int(len(all_samples) * 0.15)
train_samples = all_samples[val_count:]
val_samples = all_samples[:val_count]

train_loader = DataLoader(ITDDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(ITDDataset(val_samples), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Build Model
num_classes = len(CLASS_MAP)
model = seg_models.lraspp_mobilenet_v3_large(weights=seg_models.LRASPP_MobileNet_V3_Large_Weights.DEFAULT)
in_channels = model.classifier.low_classifier.in_channels
high_channels = model.classifier.high_classifier.in_channels
model.classifier = seg_models.lraspp.LRASPPHead(low_channels=in_channels, high_channels=high_channels, num_classes=num_classes, inter_channels=128)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float("inf")
print("Starting training on:", DEVICE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)["out"]
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            outputs = model(imgs)["out"]
            loss = criterion(outputs, masks)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = OUTPUT_DIR / "itd_segmenter.pt"
        torch.save({"state_dict": model.state_dict(), "num_classes": num_classes, "architecture": "lraspp_mobilenet_v3_large"}, checkpoint_path)
        print(f"  --> Saved Best Checkpoint to {checkpoint_path.name}")

### Step 4: Extract Prototypes & Build OpenCLIP ViT-B/32 Prototype Index
Crops distinct dish regions from ground truth masks and encodes them into dense normalized embeddings using OpenCLIP.

In [ ]:
import open_clip

print("Loading OpenCLIP ViT-B-32...")
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k",
    device=DEVICE
)
clip_model.eval()

PROTOTYPES_DIR = OUTPUT_DIR / "prototype_crops"
PROTOTYPES_DIR.mkdir(parents=True, exist_ok=True)

prototypes_per_class = 20
class_counts = {int(k): 0 for k in CLASS_MAP.keys()}
crops_metadata = []

print("Extracting high-quality dish crops from ITD masks...")
for img_file, mask_file in all_samples:
    mask = np.array(Image.open(mask_file).convert("L"))
    present_classes = set(np.unique(mask)) - {0, 255}
    if not present_classes:
        continue
    
    # Check if any needed
    needed = [c for c in present_classes if class_counts.get(c, 0) < prototypes_per_class]
    if not needed:
        continue

    with Image.open(img_file) as im:
        image = im.convert("RGB")
    
    for c in needed:
        if class_counts[c] >= prototypes_per_class:
            continue
        binary_mask = (mask == c).astype(np.uint8)
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if area < 1200: # filter tiny fragments
                continue
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            crop = image.crop((x, y, x + w, y + h))
            
            crop_name = f"class_{c}_{class_counts[c]:03d}.jpg"
            crop_path = PROTOTYPES_DIR / crop_name
            crop.save(crop_path, quality=90)
            
            crops_metadata.append({
                "classId": c,
                "className": CLASS_MAP.get(str(c), f"Class_{c}"),
                "cropPath": str(crop_path),
                "cropName": crop_name
            })
            class_counts[c] += 1
            if class_counts[c] >= prototypes_per_class:
                break

print(f"Extracted total {len(crops_metadata)} prototypes across {sum(1 for v in class_counts.values() if v > 0)} classes.")

### Step 5: Compute CLIP Embeddings and Centroids

In [ ]:
import numpy as np

embeddings = []
batch_size = 32
valid_meta = []

for i in range(0, len(crops_metadata), batch_size):
    batch = crops_metadata[i:i + batch_size]
    tensors = []
    for m in batch:
        with Image.open(m["cropPath"]) as im:
            tensors.append(preprocess(im.convert("RGB")))
    
    input_tensor = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        feats = clip_model.encode_image(input_tensor)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        embeddings.append(feats.cpu().numpy())
    valid_meta.extend(batch)

all_embeddings = np.concatenate(embeddings, axis=0)
print("Embeddings matrix shape:", all_embeddings.shape)

# Calculate class centroids
centroids = {}
for c_id in sorted(set(m["classId"] for m in valid_meta)):
    idxs = [idx for idx, m in enumerate(valid_meta) if m["classId"] == c_id]
    class_vecs = all_embeddings[idxs]
    centroid = np.mean(class_vecs, axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    centroids[str(c_id)] = {
        "className": CLASS_MAP[str(c_id)],
        "centroid": centroid.tolist(),
        "numPrototypes": len(idxs)
    }

# Save index
index_path = OUTPUT_DIR / "itd_recognition_index.json"
index_data = {
    "model": "ViT-B-32",
    "pretrained": "laion2b_s34b_b79k",
    "embeddingDim": all_embeddings.shape[1],
    "totalPrototypes": len(valid_meta),
    "classes": centroids
}
index_path.write_text(json.dumps(index_data, indent=2))
np.save(OUTPUT_DIR / "itd_recognition_index_embeddings.npy", all_embeddings)

print(f"\nIndex successfully generated and saved to {index_path.name}!")

### Step 6: Package Artifacts for Download
Zips the resulting model weights and recognition index for easy 1-click download from Kaggle.

In [ ]:
!tar -czvf /kaggle/working/nutrilens_itd_artifacts.tar.gz -C /kaggle/working/nutrilens_models itd_segmenter.pt itd_recognition_index.json iiith_itd_class_map.json itd_recognition_index_embeddings.npy

print("\n=======================================================")
print("Training Complete! Download 'nutrilens_itd_artifacts.tar.gz' from the Kaggle Output tab.")
print("=======================================================")